# Feature Engineering

This notebook prepares the features for machine learning based on the findings from the EDA. All transformations are fitted on the training data and then applied to the validation and test sets.

## 1. Load the Data

I load the training, validation, and test datasets created in Notebook 3.

In [2]:
import pandas as pd

train = pd.read_csv("../artifacts/train.csv")
validation = pd.read_csv("../artifacts/validation.csv")
test = pd.read_csv("../artifacts/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 28)
Validation: (14471, 28)
Test: (14472, 28)


## 2. Select Features

I select the features that are available at prediction time and exclude identifiers and future delivery information.

In [13]:
# Features available at prediction time

feature_cols = [

    # Order and payment information
    "item_count",
    "total_price",
    "total_freight",
    "payment_count",
    "total_payment",
    "payment_installments",

    # Product information
    "unique_products",
    "average_product_weight",
    "average_product_photos",

    # Customer / seller location
    "customer_state",
    "seller_state",
    "customer_zip_code_prefix",
    "seller_zip_code_prefix"

]

target = "delivery_label"


X_train = train[feature_cols].copy()
y_train = train[target].copy()

X_validation = validation[feature_cols].copy()
y_validation = validation[target].copy()

X_test = test[feature_cols].copy()
y_test = test[target].copy()

In [14]:
# Create safe geographical features

X_train["same_state"] = (
    X_train["customer_state"] == X_train["seller_state"]
).astype(int)

X_validation["same_state"] = (
    X_validation["customer_state"] == X_validation["seller_state"]
).astype(int)

X_test["same_state"] = (
    X_test["customer_state"] == X_test["seller_state"]
).astype(int)

In [15]:
# Check the new geographical feature

print("Train shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

print("\nSame-state distribution in training data:")
print(X_train["same_state"].value_counts())

Train shape: (67533, 14)
Validation shape: (14471, 14)
Test shape: (14472, 14)

Same-state distribution in training data:
same_state
0    44448
1    23085
Name: count, dtype: int64


In [6]:
# Check the new geographical feature

print("Train shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

print("\nSame-state distribution in training data:")
print(X_train["same_state"].value_counts())

Train shape: (67533, 14)
Validation shape: (14471, 14)
Test shape: (14472, 14)

Same-state distribution in training data:
same_state
0    44448
1    23085
Name: count, dtype: int64


## 3. Preprocessing

I handle missing values and encode categorical features using transformations fitted only on the training data.

In [3]:
#%pip install scikit-learn

In [17]:
# Preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = [
    "customer_state",
    "seller_state"
]

numerical_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# Include the engineered geographical feature
if "same_state" not in numerical_features:
    numerical_features.append("same_state")

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted transformer to validation and test
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

Preprocessing completed successfully.


## 4. Save the Preprocessor

I save the fitted preprocessing pipeline so the same transformations can be reused consistently on new data.

In [19]:
import joblib

joblib.dump(
    preprocessor,
    "../artifacts/preprocessor.joblib"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


## 5. Transform the Data

I apply the fitted preprocessing pipeline to generate the final feature matrices for training, validation, and testing.

In [20]:
# Check the final processed feature matrices

print("Train features:", X_train_processed.shape)
print("Validation features:", X_validation_processed.shape)
print("Test features:", X_test_processed.shape)

print("\nMatrix type:", type(X_train_processed))

Train features: (67533, 61)
Validation features: (14471, 61)
Test features: (14472, 61)

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


## 6. Save the Feature Data

I save the processed feature matrices and target labels as artifacts for model training.

In [21]:
# Save sparse processed feature matrices and target labels

import numpy as np
from scipy.sparse import save_npz

save_npz("../artifacts/X_train.npz", X_train_processed)
save_npz("../artifacts/X_validation.npz", X_validation_processed)
save_npz("../artifacts/X_test.npz", X_test_processed)

np.save("../artifacts/y_train.npy", y_train.to_numpy())
np.save("../artifacts/y_validation.npy", y_validation.to_numpy())
np.save("../artifacts/y_test.npy", y_test.to_numpy())

print("Sparse feature data and target labels saved successfully.")

Sparse feature data and target labels saved successfully.


## 7. Save the Feature List

I save the original feature list used for preprocessing to keep the feature definition reproducible.

In [23]:
# Save the complete feature list

final_feature_list = numerical_features + categorical_features

with open("../artifacts/feature_list.txt", "w", encoding="utf-8") as f:
    for feature in final_feature_list:
        f.write(feature + "\n")

print("Feature list saved successfully.")
print("Number of original features:", len(final_feature_list))

Feature list saved successfully.
Number of original features: 14


## 8. Verify Saved Artifacts

I verify that the required feature engineering artifacts have been created successfully.

In [24]:
# Verify saved artifacts

import os

artifacts = [
    "preprocessor.joblib",
    "X_train.npz",
    "X_validation.npz",
    "X_test.npz",
    "y_train.npy",
    "y_validation.npy",
    "y_test.npy",
    "feature_list.txt"
]

for artifact in artifacts:
    path = os.path.join("../artifacts", artifact)
    print(f"{artifact}: {os.path.exists(path)}")

preprocessor.joblib: True
X_train.npz: True
X_validation.npz: True
X_test.npz: True
y_train.npy: True
y_validation.npy: True
y_test.npy: True
feature_list.txt: True
